## EGARCH(1,1)-X (tone + article growth)

We now augment the baseline GARCH(1,1) by (i) replacing the symmetric ARCH term with an EGARCH log-variance specification that allows a sign asymmetry through the leverage parameter $\xi$, and (ii) injecting two exogenous sentiment regressors into the variance equation:

$$\ln(\sigma_t^2) \;=\; \omega \;+\; \alpha\bigl[\,|z_{t-1}| - \mathbb{E}|z_{t-1}|\bigr] \;+\; \xi\, z_{t-1} \;+\; \beta\, \ln(\sigma_{t-1}^2) \;+\; \gamma_1\, \text{Tone}_{m,t-1} \;+\; \gamma_2\, \text{artgrowth}_{m,t-1},$$

where $z_t = \varepsilon_t / \sigma_t$ is the standardized residual, with $\mathbb{E}|z_t| = \sqrt{2/\pi}$ under the Gaussian assumption. The parameters $(\mu, \omega, \alpha, \xi, \beta, \gamma_1, \gamma_2)$ are estimated by Maximum Likelihood (Gaussian innovations) with robust sandwich standard errors $H^{-1} J H^{-1}$. The results are reported in the same format as the GARCH(1,1) baseline so that log-likelihood, AIC and BIC are directly comparable.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pd.set_option('display.float_format', '{:.6f}'.format)
%matplotlib inline


In [2]:
IN_PATH = '../CLEANED DATA/REMX_prices_sentiment_combined.xlsx'

df = pd.read_excel(IN_PATH, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
df = df.set_index('Date')
df['r'] = np.log(df['Close'] / df['Close'].shift(1))
df['r'] = df['r'] - df['r'].mean()
r  = df['r'].dropna()
r2 = (r ** 2)


# Gaussian shocks

In [3]:
%%capture cap_egarch_x_normal
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm
from statsmodels.tools.numdiff import approx_hess

# ── 1. Align r_t (in %) with the contemporaneous Tone_t and art_growth_t.
#       The t-1 lag required by the EGARCH-X equation is applied inside the
#       recursion below via x[t-1] indexing — DO NOT pre-shift here.
r_pct          = df['r'] * 100.0
tone       = df['tone_mean']
art_growth = df['art_growth']

aligned = pd.concat(
    [r_pct, tone, art_growth],
    axis=1, keys=['r', 'tone', 'art_growth']
).dropna()

r          = aligned['r'].values
tone       = aligned['tone'].values
art_growth = aligned['art_growth'].values
T          = len(r)

print(f'Sample after alignment: T = {T} obs '
      f'({aligned.index.min().date()} -> {aligned.index.max().date()})')

# ── 2. EGARCH(1,1)-X recursion (returns per-obs log-likelihood) ───────────
SQRT_2_OVER_PI = np.sqrt(2.0 / np.pi)
PARAM_NAMES = ['mu', 'omega', 'alpha', 'xi', 'beta', 'gamma_tone', 'gamma_artgrowth']
CLAMP = 30.0  # numerical guard on log_sig2 during optimization

def per_obs_ll(params, r, tone, art_growth):
    mu, omega, alpha, xi, beta, g1, g2 = params
    n = len(r)
    eps = r - mu
    log_sig2 = np.empty(n)
    log_sig2[0] = np.log(max(np.var(eps), 1e-8))

    for t in range(1, n):
        z_prev = eps[t-1] / np.exp(0.5 * log_sig2[t-1])
        ls = (omega
              + alpha * (np.abs(z_prev) - SQRT_2_OVER_PI)
              + xi    * z_prev
              + beta  * log_sig2[t-1]
              + g1    * tone[t-1]
              + g2    * art_growth[t-1])
        log_sig2[t] = np.clip(ls, -CLAMP, CLAMP)

    ll_t = -0.5 * (np.log(2.0 * np.pi) + log_sig2 + eps**2 / np.exp(log_sig2))
    return ll_t, log_sig2

def neg_ll(params, r, tone, art_growth):
    ll_t, _ = per_obs_ll(params, r, tone, art_growth)
    return -ll_t.sum()

# ── 3. Maximum-likelihood estimation ──────────────────────────────────────
x0 = np.array([0.0, -0.10, 0.10, -0.05, 0.95, 0.0, 0.0])
opt = minimize(neg_ll, x0, args=(r, tone, art_growth),
               method='L-BFGS-B', options={'disp': False, 'maxiter': 5000})
if not opt.success:
    print(f'WARNING: optimizer message: {opt.message}')

theta = opt.x
ll    = -opt.fun

# ── 4. Robust (sandwich) standard errors ──────────────────────────────────
H      = approx_hess(theta, neg_ll, args=(r, tone, art_growth))
V_hess = np.linalg.inv(H)

# Numerical score matrix (per-observation gradients), central differences
eps_fd = 1e-5
G = np.empty((T, len(theta)))
for i in range(len(theta)):
    p_up = theta.copy(); p_up[i] += eps_fd
    p_dn = theta.copy(); p_dn[i] -= eps_fd
    ll_up, _ = per_obs_ll(p_up, r, tone, art_growth)
    ll_dn, _ = per_obs_ll(p_dn, r, tone, art_growth)
    G[:, i] = (ll_up - ll_dn) / (2 * eps_fd)

J        = G.T @ G                # outer product of scores (BHHH)
V_robust = V_hess @ J @ V_hess    # sandwich
se       = np.sqrt(np.diag(V_robust))
t_ratios = theta / se
p_values = 2.0 * (1.0 - norm.cdf(np.abs(t_ratios)))

# ── 5. Information criteria ───────────────────────────────────────────────
k   = len(theta)
aic = -2.0 * ll + 2.0 * k
bic = -2.0 * ll + k * np.log(T)

# ── 6. Report  ─────────────────────────────────
tbl = pd.DataFrame({
    'estimate': theta,
    'std_err':  se,
    't_ratio':  t_ratios,
    'p_value':  p_values,
}, index=PARAM_NAMES)
tbl['significant_10pct'] = tbl['t_ratio'].abs() > 1.645
tbl['significant_5pct']  = tbl['t_ratio'].abs() > 1.96
tbl['significant_1pct']  = tbl['t_ratio'].abs() > 2.576

print()
print('Per-parameter significance (|t| > 1.645 -> 10%, |t| > 1.96 -> 5%, |t| > 2.576 -> 1%):')
print(tbl.to_string(float_format=lambda x: f'{x: .4f}'))
print()
print(f'Log-likelihood : {ll: .4f}')
print(f'AIC            : {aic: .4f}')
print(f'BIC            : {bic: .4f}')
print()
beta_hat = theta[4]
print(f'beta = {beta_hat:.4f}   '
      f'({"stationary (|beta|<1)" if abs(beta_hat) < 1 else "non-stationary (|beta|>=1)"})')
half_life = np.log(0.5) / np.log(abs(beta_hat)) if 0 < abs(beta_hat) < 1 else float('inf')
print(f'Half-life of a log-variance shock: {half_life:.1f} trading days')
print()
print('Verdict:')
for name, row in tbl.iterrows():
    if row['significant_1pct']:
        tag = 'SIGNIFICANT at 1% (and 5%, 10%)'
    elif row['significant_5pct']:
        tag = 'SIGNIFICANT at 5% (and 10%) only'
    elif row['significant_10pct']:
        tag = 'SIGNIFICANT at 10% only'
    else:
        tag = 'not significant'
    print(f'  {name:<15}  t = {row["t_ratio"]:+.2f}   p = {row["p_value"]:.4f}   -> {tag}')


In [4]:
from pathlib import Path

REPORTS_DIR = Path("../REPORTS")
REPORTS_DIR.mkdir(exist_ok=True)

# Persist capture for the report
(REPORTS_DIR / "04-extension-models-gaussian.txt").write_text(cap_egarch_x_normal.stdout)

print(cap_egarch_x_normal.stdout)
print(f"Persisted capture -> REPORTS/04-extension-models-gaussian.txt")

Sample after alignment: T = 2765 obs (2015-04-02 -> 2026-03-31)

Per-parameter significance (|t| > 1.645 -> 10%, |t| > 1.96 -> 5%, |t| > 2.576 -> 1%):
                 estimate  std_err   t_ratio  p_value  significant_10pct  significant_5pct  significant_1pct
mu                -0.0270   0.0369   -0.7305   0.4651              False             False             False
omega              0.0390   0.0117    3.3348   0.0009               True              True              True
alpha              0.1696   0.0279    6.0858   0.0000               True              True              True
xi                -0.0227   0.0135   -1.6781   0.0933               True             False             False
beta               0.9786   0.0072  135.7467   0.0000               True              True              True
gamma_tone         0.0061   0.0062    0.9857   0.3243              False             False             False
gamma_artgrowth   -0.0177   0.0086   -2.0499   0.0404               True              

# Student-t shocks

In [5]:
%%capture cap_egarch_x_t
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import gammaln
from scipy.stats import norm
from statsmodels.tools.numdiff import approx_hess

# ── 1. Align r_t (in %) with the contemporaneous Tone_t and art_growth_t.
#       The t-1 lag required by the EGARCH-X equation is applied inside the
#       recursion below via x[t-1] indexing — DO NOT pre-shift here.
r_pct          = df['r'] * 100.0
tone       = df['tone_mean']
art_growth = df['art_growth']

aligned = pd.concat(
    [r_pct, tone, art_growth],
    axis=1, keys=['r', 'tone', 'art_growth']
).dropna()

r          = aligned['r'].values
tone       = aligned['tone'].values
art_growth = aligned['art_growth'].values
T          = len(r)

print(f'Sample after alignment: T = {T} obs '
      f'({aligned.index.min().date()} -> {aligned.index.max().date()})')

# ── 2. EGARCH(1,1)-X recursion with standardized Student-t innovations ───
PARAM_NAMES = ['mu', 'omega', 'alpha', 'xi', 'beta',
               'gamma_tone', 'gamma_artgrowth', 'nu']
CLAMP = 30.0  # numerical guard on log_sig2 during optimization

def _et_abs_z(nu):
    # E|z| for a standardized Student-t with nu degrees of freedom
    return (2.0 * np.sqrt(nu - 2.0)
            * np.exp(gammaln((nu + 1) / 2.0) - gammaln(nu / 2.0))
            / ((nu - 1.0) * np.sqrt(np.pi)))

def per_obs_ll(params, r, tone, art_growth):
    mu, omega, alpha, xi, beta, g1, g2, nu = params
    n = len(r)
    eps = r - mu
    log_sig2 = np.empty(n)
    log_sig2[0] = np.log(max(np.var(eps), 1e-8))

    e_abs_z = _et_abs_z(nu)

    for t in range(1, n):
        z_prev = eps[t-1] / np.exp(0.5 * log_sig2[t-1])
        ls = (omega
              + alpha * (np.abs(z_prev) - e_abs_z)
              + xi    * z_prev
              + beta  * log_sig2[t-1]
              + g1    * tone[t-1]
              + g2    * art_growth[t-1])
        log_sig2[t] = np.clip(ls, -CLAMP, CLAMP)

    # Standardized Student-t log-density (per obs)
    z2 = eps**2 / np.exp(log_sig2)
    log_const = (gammaln((nu + 1) / 2.0)
                 - gammaln(nu / 2.0)
                 - 0.5 * np.log(np.pi * (nu - 2.0)))
    ll_t = log_const - 0.5 * log_sig2 - ((nu + 1) / 2.0) * np.log1p(z2 / (nu - 2.0))
    return ll_t, log_sig2

def neg_ll(params, r, tone, art_growth):
    if params[7] <= 2.01:   # nu must be > 2 for finite variance
        return 1e10
    ll_t, _ = per_obs_ll(params, r, tone, art_growth)
    return -ll_t.sum()

# ── 3. Maximum-likelihood estimation ─────────────────────────────────────
x0 = np.array([0.0, -0.10, 0.10, -0.05, 0.95, 0.0, 0.0, 8.0])
bounds = [(None, None), (None, None), (None, None), (None, None),
          (-0.999, 0.999), (None, None), (None, None), (2.05, 200.0)]
opt = minimize(neg_ll, x0, args=(r, tone, art_growth),
               method='L-BFGS-B', bounds=bounds,
               options={'disp': False, 'maxiter': 5000})
if not opt.success:
    print(f'WARNING: optimizer message: {opt.message}')

theta = opt.x
ll    = -opt.fun

# ── 4. Robust (sandwich) standard errors ─────────────────────────────────
H      = approx_hess(theta, neg_ll, args=(r, tone, art_growth))
V_hess = np.linalg.inv(H)

eps_fd = 1e-5
G = np.empty((T, len(theta)))
for i in range(len(theta)):
    p_up = theta.copy(); p_up[i] += eps_fd
    p_dn = theta.copy(); p_dn[i] -= eps_fd
    ll_up, _ = per_obs_ll(p_up, r, tone, art_growth)
    ll_dn, _ = per_obs_ll(p_dn, r, tone, art_growth)
    G[:, i] = (ll_up - ll_dn) / (2 * eps_fd)

J        = G.T @ G                # BHHH outer product of scores
V_robust = V_hess @ J @ V_hess    # sandwich
se       = np.sqrt(np.diag(V_robust))
t_ratios = theta / se
p_values = 2.0 * (1.0 - norm.cdf(np.abs(t_ratios)))

# ── 5. Information criteria ──────────────────────────────────────────────
k   = len(theta)
aic = -2.0 * ll + 2.0 * k
bic = -2.0 * ll + k * np.log(T)

# ── 6. Report ────────────────────────────────────────────────────────────
tbl = pd.DataFrame({
    'estimate': theta,
    'std_err':  se,
    't_ratio':  t_ratios,
    'p_value':  p_values,
}, index=PARAM_NAMES)
tbl['significant_10pct'] = tbl['t_ratio'].abs() > 1.645
tbl['significant_5pct']  = tbl['t_ratio'].abs() > 1.96
tbl['significant_1pct']  = tbl['t_ratio'].abs() > 2.576

print()
print('Per-parameter significance (|t| > 1.645 -> 10%, |t| > 1.96 -> 5%, |t| > 2.576 -> 1%):')
print(tbl.to_string(float_format=lambda x: f'{x: .4f}'))
print()
print(f'Log-likelihood : {ll: .4f}')
print(f'AIC            : {aic: .4f}')
print(f'BIC            : {bic: .4f}')
print()
beta_hat = theta[4]
nu_hat   = theta[7]
print(f'beta = {beta_hat:.4f}   '
      f'({"stationary (|beta|<1)" if abs(beta_hat) < 1 else "non-stationary (|beta|>=1)"})')
half_life = np.log(0.5) / np.log(abs(beta_hat)) if 0 < abs(beta_hat) < 1 else float('inf')
print(f'Half-life of a log-variance shock: {half_life:.1f} trading days')
print(f'nu   = {nu_hat:.4f}   (Student-t dof; lower -> heavier tails. Note: the t-ratio for nu '
      f'tests H0: nu = 0, which is not the meaningful null. To test Gaussian vs Student-t use '
      f'an LR test against the Gaussian EGARCH-X.)')
print()
print('Verdict:')
for name, row in tbl.iterrows():
    if row['significant_1pct']:
        tag = 'SIGNIFICANT at 1% (and 5%, 10%)'
    elif row['significant_5pct']:
        tag = 'SIGNIFICANT at 5% (and 10%) only'
    elif row['significant_10pct']:
        tag = 'SIGNIFICANT at 10% only'
    else:
        tag = 'not significant'
    print(f'  {name:<15}  t = {row["t_ratio"]:+.2f}   p = {row["p_value"]:.4f}   -> {tag}')


In [6]:
from pathlib import Path

REPORTS_DIR = Path("../REPORTS")
REPORTS_DIR.mkdir(exist_ok=True)

(REPORTS_DIR / "04-extension-models-student.txt").write_text(cap_egarch_x_t.stdout)

print(cap_egarch_x_t.stdout)
print(f"Persisted capture -> REPORTS/04-extension-models-student.txt")


Sample after alignment: T = 2765 obs (2015-04-02 -> 2026-03-31)

Per-parameter significance (|t| > 1.645 -> 10%, |t| > 1.96 -> 5%, |t| > 2.576 -> 1%):
                 estimate  std_err   t_ratio  p_value  significant_10pct  significant_5pct  significant_1pct
mu                -0.0214   0.0368   -0.5828   0.5600              False             False             False
omega              0.0352   0.0101    3.4747   0.0005               True              True              True
alpha              0.1547   0.0241    6.4299   0.0000               True              True              True
xi                -0.0147   0.0108   -1.3666   0.1717              False             False             False
beta               0.9787   0.0063  155.3734   0.0000               True              True              True
gamma_tone         0.0103   0.0049    2.0992   0.0358               True              True             False
gamma_artgrowth   -0.0177   0.0099   -1.7926   0.0730               True             F